# Interval evaluation for PyTorch networks with linear layers and ReLU

This notebook demonstrates the prototype `intervalNets` toolbox on linear-network cases and on simple ReLU activation tests.


## Initialization

If you installed the project with `pip install -e .`, you can skip the next code cell path setup. If you are running directly from a repo checkout, the next cell adds the local `src/` directory to `sys.path` so `import intervalnets` works.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "src" / "intervalnets").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root containing src/intervalnets.")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from intervalnets import IntervalTensor, enable_interval_eval

torch.set_printoptions(precision=17)
enable_interval_eval()
print(f"Using repo root: {repo_root}")

## 1. Random linear network

A random affine network shows the general propagation path.

In [ ]:
torch.manual_seed(7)
random_model = nn.Sequential(
    nn.Linear(3, 4),
    nn.Linear(4, 2),
)
random_interval = IntervalTensor.from_bounds([0.0, -1.0, 2.0], [0.2, -0.8, 2.3])
random_output = random_model.eval(random_interval)
random_output

## 2. Zero network

Even a degenerate zero input produces a nonzero-width output interval because we expand point intervals outward to capture roundoff.

In [ ]:
zero_model = nn.Linear(3, 2)
with torch.no_grad():
    zero_model.weight.zero_()
    zero_model.bias.zero_()

zero_interval = IntervalTensor.point([0.0, 0.0, 0.0])
zero_output = zero_model.eval(zero_interval)
zero_output

## 3. Hand-computable linear network

This example is easy to verify analytically because it is purely affine.

In [ ]:
linear_model = nn.Linear(2, 1)
with torch.no_grad():
    linear_model.weight.copy_(torch.tensor([[2.0, -3.0]]))
    linear_model.bias.copy_(torch.tensor([0.5]))

linear_interval = IntervalTensor.from_bounds([1.0, 2.0], [1.5, 2.5])
linear_output = linear_model.eval(linear_interval)
linear_output

## 4. Identity-style sanity check

The network should preserve the input interval up to outward rounding when it acts like the identity map.

In [ ]:
identity_model = nn.Linear(2, 2)
with torch.no_grad():
    identity_model.weight.copy_(torch.eye(2))
    identity_model.bias.zero_()

identity_interval = IntervalTensor.from_bounds([-1.0, 4.0], [1.0, 5.0])
identity_output = identity_model.eval(identity_interval)
identity_output

## 5. ReLU test cases

These examples mirror the dedicated ReLU tests in `tests/test_pytorch.py`. Each case uses assertions and then prints a clear `PASS` message when the expected interval behavior is verified.


In [ ]:
def report_pass(name: str) -> None:
    print(f"PASS: {name}")

# ReLU test 1: strictly negative intervals map to an outward-rounded zero interval.
relu = nn.ReLU()
negative_interval = IntervalTensor.from_bounds([-3.0, -0.5], [-1.0, -0.25])
negative_output = relu.eval(negative_interval)
assert all(lower < 0.0 for lower in negative_output.lower)
assert all(upper > 0.0 for upper in negative_output.upper)
report_pass("ReLU negative interval rounds outward to zero")

# ReLU test 2: positive intervals keep their endpoint images, up to outward rounding.
positive_interval = IntervalTensor.from_bounds([0.25, 1.5], [0.5, 3.0])
positive_output = relu.eval(positive_interval)
assert positive_output.lower[0] <= 0.25
assert positive_output.upper[0] >= 0.5
assert positive_output.lower[1] <= 1.5
assert positive_output.upper[1] >= 3.0
report_pass("ReLU positive interval preserves endpoint images")

# ReLU test 3: mixed-sign intervals are clamped only at the lower endpoint.
mixed_interval = IntervalTensor.from_bounds([-2.0, -1.0], [4.0, 2.5])
mixed_output = relu.eval(mixed_interval)
assert mixed_output.lower[0] < 0.0
assert mixed_output.upper[0] >= 4.0
assert mixed_output.lower[1] < 0.0
assert mixed_output.upper[1] >= 2.5
report_pass("ReLU mixed interval clamps only the lower endpoint")

# ReLU test 4: a Linear -> ReLU -> Linear network encloses all endpoint evaluations.
relu_network = nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1))
with torch.no_grad():
    relu_network[0].weight.copy_(torch.tensor([[1.0, -2.0], [-1.0, 0.5]]))
    relu_network[0].bias.copy_(torch.tensor([0.25, -0.75]))
    relu_network[2].weight.copy_(torch.tensor([[1.5, -0.5]]))
    relu_network[2].bias.copy_(torch.tensor([0.1]))

network_interval = IntervalTensor.from_bounds([-1.0, 0.5], [2.0, 1.5])
network_output = relu_network.eval(network_interval)
candidates = []
for x1 in (-1.0, 2.0):
    for x2 in (0.5, 1.5):
        hidden_1 = max(0.0, x1 - 2.0 * x2 + 0.25)
        hidden_2 = max(0.0, -x1 + 0.5 * x2 - 0.75)
        candidates.append(1.5 * hidden_1 - 0.5 * hidden_2 + 0.1)

assert network_output.lower[0] <= min(candidates)
assert network_output.upper[0] >= max(candidates)
report_pass("Linear -> ReLU -> Linear network encloses endpoint evaluations")

print("All ReLU notebook tests passed.")


## 6. Unsupported activation

ReLU is now supported. This final example shows that unsupported nonlinear activations such as `Sigmoid` still raise a `NotImplementedError`.


In [ ]:
unsupported_model = nn.Sequential(nn.Linear(2, 2), nn.Sigmoid())
try:
    unsupported_model.eval(IntervalTensor.point([0.0, 1.0]))
except NotImplementedError as exc:
    print(type(exc).__name__, exc)
